# Data Load

In [4]:
from pathlib import Path
import pandas as pd, numpy as np, re
from IPython.display import display, HTML, Markdown

files = sorted(p for p in Path("results").glob("*.csv") if p.name != "simtime_all.csv" and not p.name.endswith("_simtime.csv"))

def mech(name):
    s = name.lower()
    if s.startswith("decentralized_"): return "FD"
    if s.startswith(("semidecentralized_","semi-decentralized_")): return "SD"
    if s.startswith(("centralizedimpprox_","centralizedproximp_","centralized-proximp_","centralized_")): return "CE"

lFD, lSD, lCE = [], [], []
for p in files:
    m = mech(p.name)
    if m == "FD": lFD.append(pd.read_csv(p))
    elif m == "SD": lSD.append(pd.read_csv(p))
    elif m == "CE": lCE.append(pd.read_csv(p))

dfFD = pd.concat(lFD, ignore_index=True) if lFD else pd.DataFrame()
dfSD = pd.concat(lSD, ignore_index=True) if lSD else pd.DataFrame()
dfCE = pd.concat(lCE, ignore_index=True) if lCE else pd.DataFrame()

# Statistical Analysis

In [39]:
import pandas as pd, numpy as np, re
from pathlib import Path
from IPython.display import display, HTML

def col(df,n):
    k={re.sub(r'[^a-z0-9]','',c.lower()):c for c in df.columns}
    for x in n:
        y=re.sub(r'[^a-z0-9]','',x.lower())
        if y in k:return k[y]

def fmt(x):
    if x=="" or pd.isna(x): return ""
    try:
        f=float(x); return str(int(f)) if f.is_integer() else f"{f:.6f}"
    except: return str(x)

def stats(df):
    if df is None or df.empty:
        return pd.DataFrame([['events_total',0],['unique_items',0],['N_agent',0],['created',0],['delivered',0],['expired',0],['loaded',0],['dropped',0],['moved',0],['failures',0],['recoveries',0],['help_msgs',0],['fake_msgs',0],['rescue_actions',0],['success_rate',np.nan],['p50_latency_s',np.nan],['p90_latency_s',np.nan]],columns=['metric','value'])
    ce=col(df,['event','evt','type']); ct=col(df,['time','timestamp','sim_time','t','event_time']); ci=col(df,['item_id','item','iid','iditem']); cr=col(df,['robot_id','robot','agent','rid','aid'])
    evt=df[ce].astype(str).str.upper(); n_events=len(df); n_items=df[ci].dropna().nunique() if ci else 0
    n_agents=df.loc[df[cr].notna() & (df[cr]!=-1),cr].nunique() if cr else 0
    counts=evt.value_counts(); g=lambda *a:int(counts.reindex([t.upper() for t in a]).fillna(0).sum())
    n_create=g('ITEM','CREATE','CREATED','SPAWN'); n_deliv=g('DELIVERY','DELIVERED','DELIVER'); n_exp=g('EXPIRE','EXPIRED')
    n_load=g('LOAD'); n_drop=g('DROP'); n_move=g('MOVE'); n_fail=g('FAIL','BROKEN'); n_recv=g('RECOVERY','RECOVERED'); n_help=g('HELP'); n_fake=g('FAKE'); n_rescue=g('RESCUE')
    if ct and ci:
        t_create=df.loc[evt.isin(['ITEM','CREATE','CREATED','SPAWN'])].groupby(ci)[ct].min()
        t_deliv=df.loc[evt.isin(['DELIVERY','DELIVERED','DELIVER'])].groupby(ci)[ct].min()
        lat=(t_deliv-t_create).dropna(); success=float((t_deliv.notna() & t_create.notna()).mean()) if len(t_create)>0 else np.nan
        p50=float(lat.quantile(0.5)) if not lat.empty else np.nan; p90=float(lat.quantile(0.9)) if not lat.empty else np.nan
    else:
        success=p50=p90=np.nan
    return pd.DataFrame([
        ['events_total',n_events],['unique_items',n_items],['N_agent',n_agents],['created',n_create],['delivered',n_deliv],
        ['expired',n_exp],['loaded',n_load],['dropped',n_drop],['moved',n_move],['failures',n_fail],['recoveries',n_recv],
        ['help_msgs',n_help],['fake_msgs',n_fake],['rescue_actions',n_rescue],['success_rate',success],
        ['p50_latency_s',p50],['p90_latency_s',p90]
    ],columns=['metric','value'])

def params_for(mech):
    sim=pd.read_csv("results/simtime_all.csv")
    a=col(sim,["arch","architecture"]); r=sim[sim[a].astype(str).str.upper().eq(mech)]
    if r.empty: return pd.DataFrame([["CoordMech",mech]],columns=["Parameter","Value"])
    r=r.tail(1).squeeze()
    cNrooms=col(sim,["nrooms"]); cNbot=col(sim,["nbot","nb"]); cNmal=col(sim,["nmal"])
    cfp=col(sim,["failprob"]); cfmp=col(sim,["commfailprob"]); carr=col(sim,["arrival","arrival_s"])
    cload=col(sim,["loaditem"]); cmove=col(sim,["moveitem"]); cdrop=col(sim,["dropitem"])
    ccomp=col(sim,["comp"]); ccomm=col(sim,["comm"]); crecov=col(sim,["recovery"]); csimt=col(sim,["simtime","sim_time"])
    gi=lambda c:(int(r[c]) if c and pd.notna(r[c]) else ""); gf=lambda c:(float(r[c]) if c and pd.notna(r[c]) else "")
    dfp=pd.DataFrame([
        ["CoordMech",mech],["T_Simulation",gi(csimt)],["N_rooms",gi(cNrooms)],["Room_Source",0],["Room_Target",-1],
        ["ItemT_spawn",gi(carr)],["Item_priority","[50%,10%,40%]"],["ItemT_expire",-1],["N_agent",gi(cNbot)],
        ["N_malicious",gi(cNmal)],["F_agent",gf(cfp)],["T_load",gf(cload)],["T_move",gf(cmove)],["T_drop",gf(cdrop)],
        ["T_comp",gf(ccomp)],["T_comm",gf(ccomm)],["T_repair",gf(crecov)],["F_msg",gf(cfmp)]
    ],columns=["Parameter","Value"])
    dfp["Value"]=dfp["Value"].map(fmt); return dfp

if 'dfFD' not in globals() or 'dfSD' not in globals() or 'dfCE' not in globals():
    files=[p for p in Path("results").glob("*.csv") if p.name != "simtime_all.csv" and not p.name.endswith("_simtime.csv")]
    def mech_from_name(n):
        s=n.lower()
        if s.startswith("decentralized_"): return "FD"
        if s.startswith(("semidecentralized_","semi-decentralized_")): return "SD"
        if s.startswith(("centralizedimpprox_","centralizedproximp_","centralized-proximp_","centralized_")): return "CE"
    lFD,lSD,lCE=[],[],[]
    for p in files:
        m=mech_from_name(p.name)
        if m=="FD": lFD.append(pd.read_csv(p))
        elif m=="SD": lSD.append(pd.read_csv(p))
        elif m=="CE": lCE.append(pd.read_csv(p))
    dfFD=pd.concat(lFD,ignore_index=True) if lFD else pd.DataFrame()
    dfSD=pd.concat(lSD,ignore_index=True) if lSD else pd.DataFrame()
    dfCE=pd.concat(lCE,ignore_index=True) if lCE else pd.DataFrame()

pFD,pSD,pCE=params_for("FD"),params_for("SD"),params_for("CE")
params_tbl=(pFD.rename(columns={"Value":"FD"})
              .merge(pSD.rename(columns={"Value":"SD"}),on="Parameter",how="outer")
              .merge(pCE.rename(columns={"Value":"CE"}),on="Parameter",how="outer"))
order=["CoordMech","T_Simulation","N_rooms","Room_Source","Room_Target","ItemT_spawn","Item_priority","ItemT_expire","N_agent","N_malicious","F_agent","T_load","T_move","T_drop","T_comp","T_comm","T_repair","F_msg"]
params_tbl["Parameter"]=pd.Categorical(params_tbl["Parameter"],categories=order,ordered=True)
params_tbl=params_tbl.sort_values("Parameter").reset_index(drop=True)

outFD=stats(dfFD).rename(columns={'value':'FD'})
outSD=stats(dfSD).rename(columns={'value':'SD'})
outCE=stats(dfCE).rename(columns={'value':'CE'})
metrics_tbl=(outFD.merge(outSD,on='metric',how='outer').merge(outCE,on='metric',how='outer'))
for c in ['FD','SD','CE']:
    if c in metrics_tbl.columns: metrics_tbl[c]=metrics_tbl[c].map(fmt)

html=f"""
<div style="display:flex; gap:24px; align-items:flex-start;">
  <div style="flex:1; min-width:420px;">
    <div style="font-weight:700; margin:0 0 8px;">Simulation Parameters — FD / SD / CE</div>
    <div style="overflow:auto; max-height:70vh;">{params_tbl.to_html(index=False)}</div>
  </div>
  <div style="flex:1; min-width:420px;">
    <div style="font-weight:700; margin:0 0 8px;">Simulation Metrics — FD / SD / CE</div>
    <div style="overflow:auto; max-height:70vh;">{metrics_tbl.to_html(index=False)}</div>
  </div>
</div>
"""
display(HTML(html))

def show_lifecycles(df, mech, k=5, seed=None, title=None):
    if df is None or df.empty: return
    ce=col(df,['event','evt','type']); ct=col(df,['time','timestamp','sim_time','t','event_time'])
    ci=col(df,['item_id','item','iid','iditem']); cr=col(df,['robot_id','robot','agent','rid','aid'])
    cimp=col(df,['itemImportance','importance','priority','prio','imp'])
    if ce is None or ct is None: return

    d=df[[c for c in [ce,ct,ci,cr,cimp] if c]].copy()
    d[ct]=pd.to_numeric(d[ct],errors='coerce')
    d=d.dropna(subset=[ct])

    if ci is None:
        is_create=d[ce].astype(str).str.upper().isin(['ITEM','CREATE','CREATED','SPAWN'])
        d['_iid']=is_create.cumsum(); ci='_iid'

    items=d[ci].dropna().unique()
    if len(items)==0: return
    rng=np.random.default_rng(seed)
    pick=rng.choice(items,size=min(k,len(items)),replace=False)

    rank={'ITEM':0,'CREATE':0,'CREATED':0,'SPAWN':0,
          'STARTLOADING':1,'START_LOADING':1,'LOAD':1,
          'MOVE':2,
          'DELIVER':3,'DELIVERED':3,'DELIVERY':3,
          'DROP':3,
          'EXPIRE':4,'EXPIRED':4,
          'FAIL':5,'RECOVERY':6,'RECOVERED':6,'HELP':7,'FAKE':7,'RESCUE':7}
    imp_map={-1:'n/a',0:'L',1:'M',2:'H'}

    blocks=[]
    for iid in pick:
        sub=d[d[ci]==iid].copy()
        sub['__evt_u']=sub[ce].astype(str).str.upper()
        sub['__evt_order']=sub['__evt_u'].map(rank).fillna(2)
        sub=sub.sort_values([ct,'__evt_order',ce]).copy()   
        evtU=sub['__evt_u']                                 
        # importance
        if cimp in sub.columns:
            vals=pd.to_numeric(sub[cimp],errors='coerce').dropna()
            if not vals.empty:
                nn=vals[vals!=-1]
                imp_val=int(nn.iloc[0]) if not nn.empty else int(vals.iloc[0])
                imp=imp_map.get(imp_val,'n/a')
            else:
                imp='n/a'
        else:
            imp='n/a'
        main_robot='n/a'
        if cr in sub.columns:
            valid=(sub[cr].notna() & (sub[cr]!=-1))
            deliv=sub.loc[valid & evtU.isin(['DELIVER','DELIVERED','DELIVERY'])]
            load =sub.loc[valid & evtU.str.contains('LOAD',na=False)]
            if not deliv.empty: main_robot=int(deliv.iloc[0][cr])
            elif not load.empty: main_robot=int(load.iloc[0][cr])
            else:
                m=sub.loc[valid,cr].mode()
                if not m.empty: main_robot=int(m.iloc[0])
        lines=[]
        for _,r in sub.iterrows():
            t=f"{float(r[ct]):.2f}"; e=str(r[ce])
            if cr in sub.columns and pd.notna(r[cr]) and r[cr]!=-1:
                lines.append(f"<li>t={t}s · {e} · robot={int(r[cr])}</li>")
            else:
                lines.append(f"<li>t={t}s · {e}</li>")
        blocks.append(
            f"<div style='margin-bottom:12px;'>"
            f"<div><b>Item {iid}</b> — importance: {imp} — Robot: {main_robot}</div>"
            f"<ul style='margin:4px 0 0 16px;'>{''.join(lines[:200])}</ul>"
            f"</div>"
        )

    display(HTML(
        f"<div style='margin-top:24px;'>"
        f"<div style='font-weight:700; margin:0 0 8px;'>{title or 'Sample item lifecycles'} — {mech} ({k} random)</div>"
        f"{''.join(blocks)}"
        f"</div>"
    ))
if 'dfFD' in globals(): show_lifecycles(dfFD, "FD", k=1, seed=None)
if 'dfSD' in globals(): show_lifecycles(dfSD, "SD", k=1, seed=None)
if 'dfCE' in globals(): show_lifecycles(dfCE, "CE", k=1, seed=None)

Parameter,FD,SD,CE
CoordMech,FD,SD,CE
T_Simulation,19,21,18
N_rooms,4,4,4
Room_Source,0,0,0
Room_Target,-1,-1,-1
ItemT_spawn,120,120,120
Item_priority,"[50%,10%,40%]","[50%,10%,40%]","[50%,10%,40%]"
ItemT_expire,-1,-1,-1
N_agent,10,10,10
N_malicious,4,4,4


# Statistical Analysis